# 07 Visualization with Matplotlib

Distinctive Matplotlib views for speaker/session patterns and model-process snapshots from DuckDB tables.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.duckdb_utils import query_df, table_exists
from utils.paths import get_db_paths, resolve_workspace
from utils.text_features import parse_speakers

plt.style.use('ggplot')

workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

sessions = query_df(input_db, 'SELECT title, day, start_time_24h, track, speakers FROM sessions_in_preprocessed')
sessions['speaker_list'] = sessions['speakers'].apply(parse_speakers)
            


In [ ]:
time_df = sessions.dropna(subset=['start_time_24h']).copy()
time_df['hour'] = time_df['start_time_24h'].str.slice(0, 2).astype(int)
hour_counts = time_df.groupby('hour').size().reindex(range(8, 19), fill_value=0)
angles = np.linspace(0, 2 * np.pi, len(hour_counts), endpoint=False)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='polar')
bars = ax.bar(angles, hour_counts.values, width=0.45, alpha=0.85)
for bar, val in zip(bars, hour_counts.values):
    bar.set_facecolor(plt.cm.viridis(val / max(hour_counts.max(), 1)))
ax.set_xticks(angles)
ax.set_xticklabels([f'{h:02d}:00' for h in hour_counts.index])
ax.set_title('Session Start-Time Rhythm (Polar)')
plt.show()

In [ ]:
rows = []
for _, r in sessions.iterrows():
    for sp in r['speaker_list']:
        rows.append({'speaker': sp, 'track': r['track']})
speaker_df = pd.DataFrame(rows)
top = speaker_df['speaker'].value_counts().head(20).index
hm = speaker_df[speaker_df['speaker'].isin(top)].pivot_table(index='speaker', columns='track', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(hm.values, aspect='auto', cmap='magma')
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels(hm.index)
ax.set_xticks(range(len(hm.columns)))
ax.set_xticklabels(hm.columns, rotation=45, ha='right')
ax.set_title('Top Speaker Track Footprint')
fig.colorbar(im, ax=ax, label='Session count')
plt.tight_layout()
plt.show()

In [ ]:
if not output_db.exists() or not table_exists(output_db, 'ml_artifacts'):
    print('Run notebooks 03-06 to populate model outputs.')
else:
    art = query_df(output_db, 'SELECT notebook, metrics_json FROM ml_artifacts')
    vals = []
    for _, r in art.iterrows():
        m = json.loads(r['metrics_json']) if r['metrics_json'] else {}
        score = m.get('weighted_f1') or m.get('holdout_f1_weighted') or m.get('silhouette') or m.get('explained_total')
        if score is not None:
            vals.append({'process': r['notebook'], 'score': float(score)})
    proc = pd.DataFrame(vals).sort_values('score')
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(proc['process'], proc['score'], color='steelblue')
    ax.set_title('ML Process Metrics Snapshot')
    plt.tight_layout()
    plt.show()
            
